In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="1BCuwUZgoxhxTp5s82YrQjcQN7YU9QBFg", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/03_00_intro.mp3"))


In [ ]:
# Setup: Run this cell first!
import torch
import torch.nn as nn
import sys
import os

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"GPU available: {gpu_name}")
    print(f"   Memory: {gpu_mem:.1f} GB")
else:
    device = torch.device('cpu')
    print("No GPU detected. DeepSpeed examples require GPU.")
    print("Go to Runtime -> Change runtime type -> GPU")

print(f"\nPython {sys.version.split()[0]}")
print(f"PyTorch {torch.__version__}")

%matplotlib inline

In [ ]:
# Install DeepSpeed (takes 2-3 minutes)
!pip install -q deepspeed
import deepspeed
print(f"DeepSpeed version: {deepspeed.__version__}")

# DeepSpeed ZeRO Training -- Vizuara

In this notebook, we will use Microsoft's DeepSpeed library to apply ZeRO optimization to real model training. You will see how a single JSON configuration controls the ZeRO stage, and we will compare memory usage and training behavior across stages.

**What you will build:** A DeepSpeed training pipeline that can switch between ZeRO stages via configuration, with memory profiling to verify the theoretical savings.

**Note:** DeepSpeed is designed for multi-GPU training. On a single-GPU Colab, we will use DeepSpeed's single-GPU mode, which still applies the optimizer partitioning logic (ZeRO-1/2/3 with world_size=1 shows the API, though memory savings require N>1 GPUs).

In [ ]:
#@title 🎧 Transition: Ds Config Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_01_ds_config_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 1. Understanding DeepSpeed Configuration

DeepSpeed is configured via a JSON dictionary. The key section for ZeRO is `zero_optimization`. Let us examine what each field means.

In [ ]:
#@title 🎧 Code Walkthrough: Ds Configs
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_02_ds_configs.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
import json

# ZeRO Stage 1 config
ds_config_stage1 = {
    "train_batch_size": 32,
    "train_micro_batch_size_per_gpu": 8,
    "gradient_accumulation_steps": 4,
    "fp16": {
        "enabled": True,
        "initial_scale_power": 16,
    },
    "zero_optimization": {
        "stage": 1,
    },
    "optimizer": {
        "type": "Adam",
        "params": {
            "lr": 1e-4,
            "betas": [0.9, 0.999],
            "eps": 1e-8,
        }
    },
    "wall_clock_breakdown": False,
}

# ZeRO Stage 2 config
ds_config_stage2 = {
    "train_batch_size": 32,
    "train_micro_batch_size_per_gpu": 8,
    "gradient_accumulation_steps": 4,
    "fp16": {
        "enabled": True,
        "initial_scale_power": 16,
    },
    "zero_optimization": {
        "stage": 2,
        "allgather_partitions": True,
        "reduce_scatter": True,
        "overlap_comm": True,
    },
    "optimizer": {
        "type": "Adam",
        "params": {
            "lr": 1e-4,
            "betas": [0.9, 0.999],
            "eps": 1e-8,
        }
    },
    "wall_clock_breakdown": False,
}

# ZeRO Stage 3 config
ds_config_stage3 = {
    "train_batch_size": 32,
    "train_micro_batch_size_per_gpu": 8,
    "gradient_accumulation_steps": 4,
    "fp16": {
        "enabled": True,
        "initial_scale_power": 16,
    },
    "zero_optimization": {
        "stage": 3,
        "param_persistence_threshold": 1e5,
        "max_live_parameters": 1e9,
        "reduce_bucket_size": 5e5,
        "prefetch_bucket_size": 5e5,
    },
    "optimizer": {
        "type": "Adam",
        "params": {
            "lr": 1e-4,
            "betas": [0.9, 0.999],
            "eps": 1e-8,
        }
    },
    "wall_clock_breakdown": False,
}

print("ZeRO Stage 1 config:")
print(json.dumps(ds_config_stage1["zero_optimization"], indent=2))
print("\nZeRO Stage 2 config:")
print(json.dumps(ds_config_stage2["zero_optimization"], indent=2))
print("\nZeRO Stage 3 config:")
print(json.dumps(ds_config_stage3["zero_optimization"], indent=2))

In [ ]:
#@title 🎧 Listen: Config Breakdown
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_03_config_breakdown.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Configuration Breakdown

**Stage 1:**
- `"stage": 1` -- partition only the optimizer states
- No extra config needed. Simplest upgrade from standard DP.

**Stage 2:**
- `"stage": 2` -- partition optimizer states AND gradients
- `"reduce_scatter": True` -- use reduce-scatter instead of allreduce for gradient communication
- `"overlap_comm": True` -- overlap communication with backward pass computation

**Stage 3:**
- `"stage": 3` -- partition everything (optimizer, gradients, AND parameters)
- `"param_persistence_threshold"` -- keep tiny parameters (< this many elements) replicated to avoid allgather overhead
- `"max_live_parameters"` -- cap on how many parameters can be gathered simultaneously (controls memory spike)
- `"prefetch_bucket_size"` -- how many parameters to prefetch during forward/backward pass

In [ ]:
#@title 🎧 Transition: Transformer Model Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_04_transformer_model_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 2. Building a Transformer Model

Let us create a small Transformer model that is representative of real training workloads.

In [ ]:
#@title 🎧 Code Walkthrough: Small Transformer
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_05_small_transformer.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
class SmallTransformer(nn.Module):
    """A small Transformer for demonstrating ZeRO."""

    def __init__(self, vocab_size=1000, d_model=256, nhead=4,
                 num_layers=4, dim_feedforward=512, max_seq_len=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_embedding = nn.Embedding(max_seq_len, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_feedforward,
            batch_first=True, dropout=0.1,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.output_head = nn.Linear(d_model, vocab_size)

        self.d_model = d_model

    def forward(self, input_ids):
        B, T = input_ids.shape
        positions = torch.arange(T, device=input_ids.device).unsqueeze(0)

        x = self.embedding(input_ids) + self.pos_embedding(positions)
        x = self.transformer(x)
        logits = self.output_head(x)
        return logits


# Create model
model = SmallTransformer()
total_params = sum(p.numel() for p in model.parameters())
print(f"Model: SmallTransformer")
print(f"Parameters: {total_params:,}")
print(f"\nTheoretical memory per GPU (mixed precision):")
print(f"  Weights (fp16):    {total_params * 2 / 1e6:.1f} MB")
print(f"  Gradients (fp16):  {total_params * 2 / 1e6:.1f} MB")
print(f"  Optimizer (fp32):  {total_params * 12 / 1e6:.1f} MB")
print(f"  Total:             {total_params * 16 / 1e6:.1f} MB")

In [ ]:
#@title 🎧 Transition: Synthetic Data Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_06_synthetic_data_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 3. Synthetic Training Data

We generate random token sequences for training. The content does not matter for our purpose -- we are focusing on memory and training mechanics.

In [ ]:
#@title 🎧 Code Walkthrough: Random Token Dataset
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_07_random_token_dataset.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
from torch.utils.data import Dataset, DataLoader


class RandomTokenDataset(Dataset):
    """Generates random token sequences for training."""

    def __init__(self, vocab_size=1000, seq_len=64, num_samples=500):
        self.vocab_size = vocab_size
        self.seq_len = seq_len
        self.num_samples = num_samples

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        tokens = torch.randint(0, self.vocab_size, (self.seq_len,))
        # Next-token prediction: input is tokens[:-1], target is tokens[1:]
        return tokens[:-1], tokens[1:]


dataset = RandomTokenDataset()
print(f"Dataset size: {len(dataset)} samples")
print(f"Sequence length: {dataset.seq_len - 1} (input) -> {dataset.seq_len - 1} (target)")

# Peek at one sample
x, y = dataset[0]
print(f"Input shape: {x.shape}, Target shape: {y.shape}")

In [ ]:
#@title 🎧 Transition: Deepspeed Training Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_08_deepspeed_training_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 4. Training with DeepSpeed

Now let us write the DeepSpeed training function. The key API differences from standard PyTorch:
- `deepspeed.initialize()` wraps the model and creates the optimizer
- `model_engine.backward(loss)` replaces `loss.backward()`
- `model_engine.step()` replaces `optimizer.step()` + `optimizer.zero_grad()`

In [ ]:
#@title 🎧 Code Walkthrough: Train Function
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_09_train_function.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import time


def train_with_deepspeed(model, ds_config, dataset, num_epochs=3, stage_name="ZeRO"):
    """
    Train a model using DeepSpeed with the given ZeRO config.

    Returns losses and memory measurements.
    """
    # For single-GPU DeepSpeed, we need to set environment variables
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = '29500'
    os.environ['RANK'] = '0'
    os.environ['LOCAL_RANK'] = '0'
    os.environ['WORLD_SIZE'] = '1'

    # Initialize DeepSpeed
    model_engine, optimizer, _, _ = deepspeed.initialize(
        model=model,
        config=ds_config,
    )

    # Create dataloader
    batch_size = ds_config['train_micro_batch_size_per_gpu']
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    losses = []
    peak_memories = []

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    start_time = time.time()

    for epoch in range(num_epochs):
        epoch_losses = []
        for batch_idx, (input_ids, targets) in enumerate(dataloader):
            input_ids = input_ids.to(model_engine.device)
            targets = targets.to(model_engine.device)

            # Forward pass
            logits = model_engine(input_ids)

            # Reshape for cross-entropy: (B*T, V) vs (B*T,)
            loss = nn.functional.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1)
            )

            # Backward + step (DeepSpeed handles gradient accumulation)
            model_engine.backward(loss)
            model_engine.step()

            epoch_losses.append(loss.item())

            # Track memory
            if torch.cuda.is_available():
                peak_mem = torch.cuda.max_memory_allocated() / 1e6
                peak_memories.append(peak_mem)

        avg_loss = np.mean(epoch_losses)
        losses.append(avg_loss)
        elapsed = time.time() - start_time
        print(f"  [{stage_name}] Epoch {epoch+1}/{num_epochs}: "
              f"loss={avg_loss:.4f}, time={elapsed:.1f}s")

    # Cleanup DeepSpeed
    deepspeed.comm.destroy_process_group()

    result = {
        'losses': losses,
        'peak_memory_mb': max(peak_memories) if peak_memories else 0,
        'time_s': time.time() - start_time,
    }

    return result


print("Training function defined.")

In [ ]:
#@title 🎧 Transition: Running Stages Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_10_running_stages_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 5. Running All Three Stages

Let us train with each ZeRO stage and compare.

In [ ]:
#@title 🎧 Code Walkthrough: Run All Stages
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_11_run_all_stages.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
results = {}
num_epochs = 3

for stage_name, config in [
    ('ZeRO-1', ds_config_stage1),
    ('ZeRO-2', ds_config_stage2),
    ('ZeRO-3', ds_config_stage3),
]:
    print(f"\n{'='*50}")
    print(f"Training with {stage_name}")
    print(f"{'='*50}")

    # Fresh model for each run
    torch.manual_seed(42)
    model = SmallTransformer()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    result = train_with_deepspeed(
        model, config, dataset, num_epochs=num_epochs, stage_name=stage_name
    )
    results[stage_name] = result

    print(f"  Peak memory: {result['peak_memory_mb']:.0f} MB")
    print(f"  Total time: {result['time_s']:.1f}s")

In [ ]:
#@title 🎧 Transition: Comparing Results Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_12_comparing_results_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 6. Comparing Results

Let us visualize the training curves and memory usage.

In [ ]:
#@title 🎧 What to Look For: Visualization
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_13_visualization.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {'ZeRO-1': '#FF9800', 'ZeRO-2': '#4CAF50', 'ZeRO-3': '#2196F3'}

# Loss curves
ax = axes[0]
for name, result in results.items():
    ax.plot(range(1, num_epochs + 1), result['losses'],
            'o-', label=name, color=colors[name], linewidth=2, markersize=8)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Training Loss', fontsize=12)
ax.set_title('Training Loss by ZeRO Stage', fontsize=14)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

# Memory comparison
ax = axes[1]
names = list(results.keys())
memories = [results[n]['peak_memory_mb'] for n in names]
bars = ax.bar(names, memories, color=[colors[n] for n in names], width=0.5)
for bar, mem in zip(bars, memories):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
            f'{mem:.0f} MB', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_ylabel('Peak GPU Memory (MB)', fontsize=12)
ax.set_title('Peak Memory by ZeRO Stage', fontsize=14)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
#@title 🎧 Listen: Important Note
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_14_important_note.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Important Note

On a single GPU (world_size=1), ZeRO stages 1-3 will show similar memory because there is only one rank to partition across. The real savings appear with multiple GPUs (N>1), where each rank stores 1/N of the partitioned states. The purpose of this exercise is to learn the DeepSpeed API, verify that training converges correctly, and understand the configuration options.

In [ ]:
#@title 🎧 Transition: Offload Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_15_offload_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 7. ZeRO-Offload Configuration

For extremely large models or limited GPU memory, DeepSpeed can offload optimizer states and even parameters to CPU memory.

In [ ]:
#@title 🎧 Code Walkthrough: Offload Config
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_16_offload_config.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# ZeRO-3 with CPU offload config
ds_config_offload = {
    "train_batch_size": 32,
    "train_micro_batch_size_per_gpu": 8,
    "gradient_accumulation_steps": 4,
    "fp16": {
        "enabled": True,
        "initial_scale_power": 16,
    },
    "zero_optimization": {
        "stage": 3,
        "offload_optimizer": {
            "device": "cpu",
            "pin_memory": True,
        },
        "offload_param": {
            "device": "cpu",
            "pin_memory": True,
        },
        "param_persistence_threshold": 1e5,
        "max_live_parameters": 1e9,
    },
    "optimizer": {
        "type": "Adam",
        "params": {
            "lr": 1e-4,
            "betas": [0.9, 0.999],
        }
    },
}

print("ZeRO-3 + Offload config:")
print(json.dumps(ds_config_offload["zero_optimization"], indent=2))
print("\nKey difference: offload_optimizer and offload_param are set to 'cpu'.")
print("This moves optimizer states and parameters to CPU memory,")
print("leaving GPU memory free for activations and computation.")

In [ ]:
#@title 🎧 Code Walkthrough: Train With Offload
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_17_train_with_offload.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Train with CPU offload
print("Training with ZeRO-3 + CPU Offload")
print("="*50)

torch.manual_seed(42)
model = SmallTransformer()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

offload_result = train_with_deepspeed(
    model, ds_config_offload, dataset, num_epochs=3, stage_name="ZeRO-3+Offload"
)

print(f"\nPeak GPU memory: {offload_result['peak_memory_mb']:.0f} MB")
print(f"Total time: {offload_result['time_s']:.1f}s")

if 'ZeRO-3' in results:
    z3_mem = results['ZeRO-3']['peak_memory_mb']
    off_mem = offload_result['peak_memory_mb']
    print(f"\nGPU memory reduction from offload: {z3_mem - off_mem:.0f} MB")
    print(f"  ({(1 - off_mem / z3_mem) * 100:.1f}% less GPU memory)")
    z3_time = results['ZeRO-3']['time_s']
    off_time = offload_result['time_s']
    print(f"\nTime overhead: {off_time - z3_time:.1f}s more ({off_time/z3_time:.2f}x)")

In [ ]:
#@title 🎧 Before You Start: Todo Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_18_todo_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 8. Your Turn -- TODO Exercises

### TODO 1: Create a Config for a Larger Model

Suppose you have a 1B parameter model and 4 A100-80GB GPUs. Create DeepSpeed configs for ZeRO-2 and ZeRO-3 with appropriate settings.

In [ ]:
#@title 🎧 Before You Start: Todo
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_19_todo_1.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# ============ TODO ============
# Create a DeepSpeed config for a 1B parameter model on 4 A100 GPUs.
#
# ZeRO-2 config:
# - train_batch_size should be micro_batch * world_size * grad_accum
# - Pick a micro_batch_size of 4 (per GPU)
# - Use gradient_accumulation_steps of 8
# - Enable fp16
# - ZeRO stage 2 with reduce_scatter and overlap_comm enabled

ds_config_1b_z2 = {
    "train_batch_size": ???,           # YOUR CODE: 4 * 4 * 8 = ?
    "train_micro_batch_size_per_gpu": ???,
    "gradient_accumulation_steps": ???,
    "fp16": {
        "enabled": ???,                # YOUR CODE
        "initial_scale_power": 16,
    },
    "zero_optimization": {
        "stage": ???,                  # YOUR CODE
        "allgather_partitions": True,
        "reduce_scatter": ???,         # YOUR CODE
        "overlap_comm": ???,           # YOUR CODE
    },
    "optimizer": {
        "type": "Adam",
        "params": {
            "lr": 3e-4,
            "betas": [0.9, 0.95],
        }
    },
}

# ZeRO-3 config with offload for the same model:
ds_config_1b_z3 = {
    "train_batch_size": ???,
    "train_micro_batch_size_per_gpu": ???,
    "gradient_accumulation_steps": ???,
    "fp16": {"enabled": True},
    "zero_optimization": {
        "stage": ???,                  # YOUR CODE
        "offload_optimizer": {
            "device": ???,             # YOUR CODE: "cpu" or "none"?
            "pin_memory": True,
        },
        "param_persistence_threshold": 1e5,
    },
    "optimizer": {
        "type": "Adam",
        "params": {"lr": 3e-4, "betas": [0.9, 0.95]}
    },
}
# ==============================

print("ZeRO-2 config for 1B model:")
print(json.dumps(ds_config_1b_z2, indent=2))
print("\nZeRO-3 config for 1B model:")
print(json.dumps(ds_config_1b_z3, indent=2))

In [ ]:
#@title 🎧 Before You Start: Todo Verify
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_20_todo_1_verify.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Verification for TODO 1
assert ds_config_1b_z2['train_batch_size'] == 128, (
    f"Batch size should be 4*4*8=128, got {ds_config_1b_z2['train_batch_size']}")
assert ds_config_1b_z2['zero_optimization']['stage'] == 2
assert ds_config_1b_z3['zero_optimization']['stage'] == 3
assert ds_config_1b_z3['zero_optimization']['offload_optimizer']['device'] == 'cpu'
print("TODO 1 passed!")

In [ ]:
#@title 🎧 Before You Start: Todo Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_21_todo_2_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### TODO 2: Compute Theoretical Memory for the 1B Model

Using the formulas from Notebook 1, compute the theoretical per-GPU memory for the 1B model on 4 GPUs under each ZeRO stage.

In [ ]:
#@title 🎧 Before You Start: Todo
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_22_todo_2.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
psi_1b = 1e9  # 1 billion parameters
n_gpus = 4

def bytes_to_gb(b):
    return b / (1024 ** 3)

# ============ TODO ============
# Compute per-GPU memory for each approach.
# Formulas:
#   Standard DP: 16 * psi bytes
#   ZeRO-1: (4 * psi + 12 * psi / N) bytes
#   ZeRO-2: (2 * psi + 14 * psi / N) bytes
#   ZeRO-3: (16 * psi / N) bytes

dp_gb = ???     # YOUR CODE
z1_gb = ???     # YOUR CODE
z2_gb = ???     # YOUR CODE
z3_gb = ???     # YOUR CODE

# ==============================

print(f"Theoretical per-GPU memory for 1B model on {n_gpus} GPUs:")
print(f"  Standard DP: {dp_gb:.1f} GB")
print(f"  ZeRO-1:      {z1_gb:.1f} GB")
print(f"  ZeRO-2:      {z2_gb:.1f} GB")
print(f"  ZeRO-3:      {z3_gb:.1f} GB")
print(f"\nA100-80GB capacity: 80 GB")
print(f"  DP fits?    {'YES' if dp_gb < 80 else 'NO'}")
print(f"  ZeRO-1 fits? {'YES' if z1_gb < 80 else 'NO'}")
print(f"  ZeRO-2 fits? {'YES' if z2_gb < 80 else 'NO'}")
print(f"  ZeRO-3 fits? {'YES' if z3_gb < 80 else 'NO'}")

In [ ]:
#@title 🎧 Before You Start: Todo Verify
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_23_todo_2_verify.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Verification for TODO 2
assert abs(dp_gb - bytes_to_gb(16 * psi_1b)) < 0.1, f"DP memory wrong: {dp_gb:.1f} GB"
assert abs(z3_gb - bytes_to_gb(16 * psi_1b / n_gpus)) < 0.1, f"ZeRO-3 memory wrong: {z3_gb:.1f} GB"
assert z1_gb < dp_gb, "ZeRO-1 should use less memory than DP"
assert z3_gb < z2_gb < z1_gb < dp_gb, "Memory should decrease: DP > Z1 > Z2 > Z3"
print("TODO 2 passed!")

In [ ]:
#@title 🎧 Before You Start: Todo Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_24_todo_3_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### TODO 3: Decision Matrix

Create a function that recommends the optimal ZeRO stage given a model size, number of GPUs, and available GPU memory.

In [ ]:
#@title 🎧 Before You Start: Todo
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_25_todo_3.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def recommend_zero_stage(psi, n_gpus, gpu_memory_gb, activation_fraction=0.3):
    """
    Recommend the best ZeRO stage.

    Args:
        psi: number of parameters
        n_gpus: number of GPUs
        gpu_memory_gb: per-GPU memory in GB
        activation_fraction: fraction of GPU memory reserved for activations

    Returns:
        dict with 'recommended_stage', 'reason', and memory details
    """
    available = gpu_memory_gb * (1 - activation_fraction)

    # ============ TODO ============
    # Compute memory for each stage
    # Return the LEAST aggressive stage that fits (prefer lower comm overhead)
    #
    # Logic:
    # 1. If Standard DP fits -> recommend it (no overhead)
    # 2. Else if ZeRO-1 fits -> recommend it (no comm overhead)
    # 3. Else if ZeRO-2 fits -> recommend it (no extra comm overhead)
    # 4. Else if ZeRO-3 fits -> recommend it (1.5x comm)
    # 5. Else -> recommend ZeRO-3 + Offload

    dp_mem = bytes_to_gb(16 * psi)
    z1_mem = bytes_to_gb(4 * psi + 12 * psi / n_gpus)
    z2_mem = bytes_to_gb(2 * psi + 14 * psi / n_gpus)
    z3_mem = bytes_to_gb(16 * psi / n_gpus)

    if dp_mem <= available:
        stage = ???    # YOUR CODE
        reason = ???   # YOUR CODE
    elif z1_mem <= available:
        stage = ???
        reason = ???
    elif z2_mem <= available:
        stage = ???
        reason = ???
    elif z3_mem <= available:
        stage = ???
        reason = ???
    else:
        stage = "ZeRO-3 + Offload"
        reason = "Model states exceed GPU memory even with ZeRO-3; offload to CPU needed."

    # ==============================

    return {
        'recommended_stage': stage,
        'reason': reason,
        'memories_gb': {
            'Standard DP': dp_mem,
            'ZeRO-1': z1_mem,
            'ZeRO-2': z2_mem,
            'ZeRO-3': z3_mem,
        },
        'available_gb': available,
    }


# Test cases
test_cases = [
    (1e9, 8, 80, "1B on 8x A100-80GB"),
    (7e9, 8, 80, "7B on 8x A100-80GB"),
    (70e9, 64, 80, "70B on 64x A100-80GB"),
    (175e9, 64, 80, "175B on 64x A100-80GB"),
]

for psi, n, mem, desc in test_cases:
    result = recommend_zero_stage(psi, n, mem)
    print(f"{desc}:")
    print(f"  Recommended: {result['recommended_stage']}")
    print(f"  Reason: {result['reason']}")
    print()

In [ ]:
#@title 🎧 Before You Start: Todo Verify
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_26_todo_3_verify.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Verification for TODO 3
r1 = recommend_zero_stage(1e9, 8, 80)
assert 'DP' in r1['recommended_stage'] or 'ZeRO-1' in r1['recommended_stage'], (
    f"1B on 8 GPUs should fit with DP or ZeRO-1, got {r1['recommended_stage']}")

r2 = recommend_zero_stage(175e9, 64, 80)
assert 'ZeRO-3' in r2['recommended_stage'], (
    f"175B on 64 GPUs should need ZeRO-3, got {r2['recommended_stage']}")

print("TODO 3 passed!")

In [ ]:
#@title 🎧 Transition: Summary Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_27_summary_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 9. Summary: DeepSpeed ZeRO Quick Reference

Here is a cheat sheet for using ZeRO in practice.

In [ ]:
#@title 🎧 Listen: Quick Reference
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_28_quick_reference.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
print("""
=== DeepSpeed ZeRO Quick Reference ===

STAGE 1 - Partition optimizer states
  Config:  {"stage": 1}
  Memory:  4*psi + 12*psi/N bytes per GPU
  Comm:    Same as standard DP (1.0x)
  Use when: Model fits but you want more headroom

STAGE 2 - Partition optimizer + gradients
  Config:  {"stage": 2, "reduce_scatter": true, "overlap_comm": true}
  Memory:  2*psi + 14*psi/N bytes per GPU
  Comm:    Same as standard DP (1.0x)
  Use when: Sweet spot for most training runs

STAGE 3 - Partition everything
  Config:  {"stage": 3, "param_persistence_threshold": 1e5}
  Memory:  16*psi/N bytes per GPU
  Comm:    1.5x standard DP
  Use when: Weights don't fit on one GPU

OFFLOAD - Move states to CPU/NVMe
  Config:  {"stage": 3, "offload_optimizer": {"device": "cpu"}}
  Memory:  Minimal GPU usage
  Comm:    CPU<->GPU transfer overhead
  Use when: Nothing else fits

RULE OF THUMB:
  Start with ZeRO-2. Only move to ZeRO-3 if OOM.
  Only use offload as a last resort.
""")

In [ ]:
#@title 🎧 Wrap-Up: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_29_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 10. Reflection and Next Steps

### Think About This
1. Why is ZeRO-2 the recommended default? What is the practical advantage over ZeRO-3?
2. When would CPU offload make training faster (despite the CPU-GPU transfer overhead)?
3. How does `overlap_comm` in ZeRO-2 hide communication latency? What are its requirements?

### What Comes Next
ZeRO solves the memory redundancy problem in data parallelism. But it still treats each layer as an indivisible computation unit. In the next pod on **Tensor Parallelism**, we will learn how to split individual layers across GPUs, reducing both memory and computation per GPU.

### Key Takeaway
DeepSpeed makes ZeRO optimization remarkably accessible: a single JSON config field (`"stage": 1/2/3`) is all it takes to switch between optimization levels. The training code itself barely changes -- `model_engine.backward(loss)` and `model_engine.step()` handle all the partitioning, communication, and weight reconstruction automatically.